## Build a model from Python

In this example a simple **SFINCS** compound flood model will be made, using the underlying Python functions of **HydroMT-SFINCS** to build a model.

The model is situated in **Northern Italy**, where a small selection of topography and bathymetry data has already been made available for you to try the examples.

In [ ]:
import numpy as np
import os

from hydromt_sfincs import SfincsModel, DATADIR
from hydromt._utils import log

# Initialize logging, the lower the log level number, the more verbose (more info) the output
# NOTSET=0-9, DEBUG=10, INFO=20, WARNING=30, ERROR=40, CRITICAL=50

log.initialize_logging()
log.set_log_level(log_level=20)

This example shows how to build a SFINCS model containing elevation data and spatially varying roughness (together processed into subgrid tables), spatially varying infiltration and a local floodwall. In addition, multiple forcing conditions are set-up, and this is all done using Python scripting.

In case you want to adjust this example to build a SFINCS model anywhere else in the world, you will have to add your own datasets to HydroMT's data catalog. For more info on that, check-out:

- [Prepare data catalog](https://deltares.github.io/hydromt/stable/guides/advanced_user/data_prepare_cat.html)

- [Example: Prepare data catalog](https://deltares.github.io/hydromt/stable/_examples/prep_data_catalog.html)

**Steps followed in this notebook to build your SFINCS model:**
<ol> 
<li> Open SfincsModel class, set data library and output folder </li>
<li> Specify characteristics of the wanted grid </li>
<li> Load in wanted elevation datasets </li>
<li> Make mask of active and inactive cells </li>
<li> Update mask with water level and outflow boundary cells</li>
<li> Add intial water level conditions</li>
<li> Add spatially varying roughness data</li>
<li> Make subgrid derived tables</li>
<li> Add spatially varying infiltration data</li>
<li> Write static model to disk</li>
</ol> 

Let's get started!

### 1. Initialize SfincsModel class, set data library and output folder:

Before we can use all the tools provided by HydroMT-SFINCS, we have to initialize the SfincsModel instance. This creates a shortcut to all the model components and methods to read, write and create these components. Furthermore, when initializing the model, we can specify which data catalog we use and where the model is/will be stored (the root).

In [ ]:
# Initialize SfincsModel Python class with the artifact data catalog which contains publically available data for North Italy
sf = SfincsModel(
    data_libs=["artifact_data"],  # specify which data libraries to use
    root="tmp_sfincs_compound",  # specify the root directory for the model
    mode="w+",  # specify the mode for opening the model (r=read only, r+=append, w=write, w+=overwrite
    write_gis=True,  # specify whether to write GIS data
)

### 2. Specify characteristics of the wanted grid and generate grid:

The easiest way in HydroMT-SFINCS to generate a grid is by starting from a region and a resolution. This region can be a bounding box, or a shapefile/geojson that for example describes a watershed or administrative boundary. 

For more info about how to define a grid, [click here](https://sfincs.readthedocs.io/en/latest/input.html#grid-characteristics). 

In [ ]:
sf.grid.create_from_region(
    region={"geom": "data/region.geojson"},  # specify the region geometry
    res=50,  # specify the resolution
    rotated=True,  # specify whether the grid is rotated
    crs="utm",  # specify the coordinate reference system, when "utm", the UTM zone is automatically detected
)

The grid that we just created is described in the SFINCS configuration with x0, y0, dx, dy, mmax, nmax, rotation and epsg. In case you would have known these parameters before, you can also create a grid using `SfincsModel.grid.create()`.

In [ ]:
# the input file is automatically updated. See:
sf.config.data.model_dump()

HydroMT-SFINCS has so very usefull tools to plot your model and all of its components. The main functionality to plot the model can be found in `SfincsModel.plot_basemap()` where you can specify which background map (`bmap`) to use, and which layers to plot. Throughout this notebook, we will explore the different options of this method.

In [ ]:
# show the model grid outline
_ = sf.plot_basemap(plot_region=True, bmap="sat", zoomlevel=12)

### 3. Add elevation to the model

Probably the most important datasets to create a proper SFINCS model are the digital elevation models (DEMs). However, often there are different datasets for describing the elevation on land, topography, and below water, bathymetry. HydroMT-SFINCS provides functionality to merge these different datasets onto the model grid.

In [ ]:
# In this example we want to combine 2 elevation datasets, merit_hydro as elevation and gebco as bathymetry, in that order.
# NOTE: from the 1st dataset (merit_hydro) only elevation above ("zmin":0.001) meters is used;
# the 2nd elevation dataset (gebco) is used where the 1st dataset returned nodata values
elevation_sets = [{"elevation": "merit_hydro", "zmin": 0.001}, {"elevation": "gebco"}]

# Add depth information to modelgrid based on these chosen datasets
sf.elevation.create(elevation_sets=elevation_sets, buffer_cells=1)

# Make a plot of the merged topobathy, here colour limits are set between an elevation of -5 to 5 meters
_ = sf.plot_basemap(variable="dep", plot_region=True, bmap="sat", zoomlevel=12)

### 4. Make mask of active and inactive cells:

The next step of building a SFINCS model is to create a mask. First we generate a mask of active (msk=1) and inactive cells (msk=0), determining in which cells of your computational grid water levels will be computed.

For more info about the msk-file, [click here](https://sfincs.readthedocs.io/en/latest/input.html#mask-file). 

In [ ]:
# Defining your active cells can be based on multiple criteria, here we only specify a minimum elevation of -5 meters,
# meaning all cells above -5 are considered to be active. Check-out the method documentation for more options.
sf.mask.create_active(zmin=-5)

# # Make a plot of the mask file
_ = sf.plot_basemap(
    variable="mask", plot_region=True, plot_bounds=True, bmap="sat", zoomlevel=12
)

NOTE:
- The given output of HydroMT says "1 gap outside valid elevation range < 10 km2". 
HydroMT does some smart filtering that if small groups of inactive cells are found, surrounded by active cells, these are still included as active, in this case 1 gap.
You can control the size of these gaps to filter by adding `fill_area = 10` in `SfincsModel.mask.create_active()`.
- A similar argument exists to neglect a group of active cells surrounded by inactive cells: `drop_area`

### 5. Update mask with water level and outflow boundary cells - including use of polygons:

In SFINCS you can specify cells where you want to force offshore water levels (mask=2), or outflow boundaries (mask=3). To determine these cells, there are again multiple criteria to use, such as elevation or user-defined-polygons. Below we present both.

In [ ]:
# Here we add water level cells along the coastal boundary, for cells up to an elevation of -5 meters
sf.mask.create_boundary(
    btype="waterlevel",
    zmax=-5,
    reset_bounds=True,
)

In [ ]:
# Here we add outflow cells, only where clicked in shapefile along part of the lateral boundaries
gdf_include = sf.data_catalog.get_geodataframe(
    "data//compound_example_outflow_boundary_polygon.geojson"
)
sf.mask.create_boundary(
    btype="outflow",
    include_polygon=gdf_include,
    reset_bounds=True,
)

In [ ]:
# Make a plot of the mask file
fig, ax = sf.plot_basemap(
    variable="mask", plot_region=False, plot_bounds=False, bmap="sat", zoomlevel=12
)
# plot the shapefile given by the user as dashed line
gdf_include.to_crs(sf.crs).boundary.plot(ax=ax, color="k", lw=1, ls="--")

NOTE:
- As you can see now, also msk=2 values (red line) have been added along the coastal boundary
- As you can see now, also msk=3 values (purple line) have been added along the lateral inland boundaries within the gdf_include shapefile
- `reset_bounds=True` means that you start without initial boundary cells (of the specified type), if `reset_bounds=False` (default) you build on the existing boundary cells (if available)

### 6. Add initial water level conditions:

Wihtout any further action, SFINCS starts with initial water levels equal to `zsini` in the configuration file. In some cases, it might be usefull to start your model with some initial water levels.

In [ ]:
# Make a similar dataset as dep, but only where <=0, otherwise nodata
ini = sf.grid.data["dep"].copy()
ini.values[ini.values > 0] = ini.raster.nodata  # set all initial water
# levels to nodata where elevation > 0
ini.values[
    ini.values <= 0
] = 0  # set all initial water levels to -0.5 m where elevation <= 0

In [ ]:
# Add the initial conditions to the model and plot. For cells with initial water levels of -9999.0,
# the SFINCS kernel will set the initial water level to the bed level.

sf.initial_conditions.create(
    ini=ini,
    reproj_method="average",
)
fig, ax = sf.plot_basemap(variable="ini", plot_bounds=False, bmap="sat", zoomlevel=12)

### 6. Add river inflow points:

Based on hydrography data, a minimum river length and minimum upstream area, we can derive where rivers flow into the model. This is quite a slow function since flow-directions are being derived from the hydrography. The rivers derived by this method are not used by SFINCS, but discharge points are added at the inflow points (with a dummy discharge).

In [ ]:
sf.rivers.create_river_inflow(
    hydrography="merit_hydro",
    river_len=1000,
    river_upa=50,
    keep_rivers_geom=True,
    merge=False,
)

# Make a plot of model
# note the src points and derived river network (the latter is just for visualization and not used by SFINCS)
fig, ax = sf.plot_basemap(variable="mask", plot_bounds=False, bmap="sat", zoomlevel=12)

### 7. Add spatially varying roughness data:

Next to the elevation, the roughness is also an important input dataset for the SFINCS model. Typically, the mannings' n roughness coefficients are derived based on landuse/landcover data and a conversion table.

To ensure a correct lower roughness of the rivers in the domain, we can also 'burn in' the river with a lower manning roughness value. There is two different ways to do this:
<ol>
<li>Rasterize the manning value of geodataframe to the model grid </li>
<li>Or directly burn the roughness into the subgrid tables (see below)</li>
</ol>

In [ ]:
# At first we create a geodataframe with river attributes based on the derived rivers
# Each river has a centerline (derived before) and a width, depth and manning
gdf_riv = sf.rivers.data.copy(deep=True)
gdf_riv["rivwth"] = [100, 50, 50]  # width [m]
gdf_riv["rivdph"] = 1.5  # depth [m]
gdf_riv["manning"] = 0.03  # manning coefficient [s.m-1/3]

In [ ]:
# 1. rasterize the manning value of gdf to the  model grid and use this as a manning raster
gdf_riv_buf = gdf_riv.assign(geometry=gdf_riv.buffer(gdf_riv["rivwth"] / 2))
da_manning = sf.grid.data.raster.rasterize(gdf_riv_buf, "manning", nodata=np.nan)

# uncomment to plot either the raster or the vector data:
# da_manning.plot(vmin=0, x="xc", y="yc", cmap="viridis")

In [ ]:
# Some reclass tables are available in the hydromt_sfincs DATADIR
# e.g. vito_mapping.csv for vito_2015 landuse classification
reclass_table = os.path.join(DATADIR, "lulc", "vito_mapping.csv")

# use the river manning raster in combination with vito_2015 land to derive the manning roughness file
roughness_sets = [
    {"manning": da_manning},
    {"lulc": "vito_2015", "reclass_table": reclass_table},
]

sf.roughness.create(
    roughness_sets=roughness_sets,
    manning_land=0.04,
    manning_sea=0.02,
    rgh_lev_land=0,  # the minimum elevation of the land
)
_ = sf.plot_basemap(variable="manning", plot_bounds=False, bmap="sat", zoomlevel=12)

### 8. Make subgrid derived tables:

Subgrid derived tables are used to better capture the elevation and roughness of your domain, to either improve your results, or to allow you to run on a courser grid resolution (means faster simulation). For more info about subgrid tables, [click here](https://sfincs.readthedocs.io/en/latest/developments.html#recent-advancements-in-accuracy-subgrid-mode). 

You as user can specify multiple settings about how the subgrid derived tables should be made.

Every single grid cell of the flux grid of the size inp.dx by inp.dy is defined into subgrid pixels (default nr_subgrid_pixels = 20).
For every subgrid pixel the topobathy data is loaded, ideally this consists of high-resolution DEM datasets that you specify as user.

In this example with dx=dy=50m, having nr_subgrid_pixels = 20 means we are loading data onto a 2.5 m subpixel grid
However, the input data of Gebco and Merit_hydro is way coarser, therefore let's set the ratio to 6 for now.

    
<div style="border-left: 4px solid #4CAF50; padding: 0.5em; background-color: #f0fff0;">
<b>⚠️ Note:</b> The subgrid tables replace the elevation and roughness data that we prepared before!
</div>

For the subgrid tables, we again provide the elevation datasets, and roughness datasets. Additonally, we can provide river datasets that are used to burn in rivers into the elevation and roughness maps. 

In [ ]:
# Repeat elevation sets for clarity
elevation_sets = [{"elevation": "merit_hydro", "zmin": 0.001}, {"elevation": "gebco"}]

# Roughness sets, now without the rasterized manning values
roughness_sets = [{"lulc": "vito_2015", "reclass_table": reclass_table}]

# River sets with centerlines and attributes
river_sets = [{"centerlines": gdf_riv}]

sf.subgrid.create(
    elevation_sets=elevation_sets,
    roughness_sets=roughness_sets,
    river_sets=river_sets,
    nr_subgrid_pixels=6,
    write_dep_tif=True,
    write_man_tif=True,
)

# NOTE: we turned on that the merged topobathy of the different (high-res) datasets is written to a geotiff

<div style="border-left: 4px solid #4CAF50; padding: 0.5em; background-color: #f0fff0;">
<b>⚠️ Note:</b> If you have a very large domain with more than a millions cells, and very high-resolution datasets (e.g. 1 m), this step might take minutes to hours!!! 

But good news; when finished succesfully, you can very quickly run very accurate SFINCS simulations! The whole point of the subgrid functionality of SFINCS is that by derived subgrid tables based on high res elevation data, you either have more accurate results or run on a coarser grid resolution (= much faster) or both.
</div>


Now we can see what kind of subgrid-derived variables are created:

In [ ]:
# uncomment to see the subgrid table variales
sf.subgrid.data

# we can plot the 2D subgrid variables
_ = sf.plot_basemap(
    variable=sf.subgrid.data["u_navg"], plot_bounds=False, bmap="sat", zoomlevel=12
)

### 9. Add spatially varying infiltration data:

SFINCS (and HydroMT-SFINCS) contains a couple of different methods to specify infiltration. In the example below, the curve number method is used, but we recommend to checkout the documentation to explore other options.

In [ ]:
# independent from subgrid files
# curve number infiltration based on global CN dataset
sf.infiltration.create_cn("gcn250", antecedent_moisture="avg")

_ = sf.plot_basemap(variable="scs", plot_bounds=False, bmap="sat", zoomlevel=12)

### 10. Write the (static) model input to disk

By now we have made all basic SFINCS spatial layers to make the mskfile, infiltrationfile and subgridfiles. These layers are often independent on the scenario and can be reused for multiple simulations. It's therefore usefull to write the model to disk at this stage.

In [ ]:
sf.write()

Nonetheless, we are not yet done yet with building a model. Before we can run the model, we should add boundary conditions to the model, and eventually add some observation points or levees to the model. This is demonstrated in the following notebooks:
- [2. Add Forcing](2_add_forcing.ipynb)
- [3. Add Geometries](3_add_geometries.ipynb)

Once you have finished your model, you can progress to the notebook: 
- [4. Run the model](4_run_model.ipynb)